# model-save-state-dict — ex2: atomic rank-0 save via tmp + os.replace + barrier

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `model-save-state-dict`. Running the final beacon cell reports progress against the `Distributed: model save state_dict rank-0` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Distributed: model save state_dict rank-0` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`model-save-state-dict`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "model-save-state-dict"
DD_SUBTOPIC = "Distributed: model save state_dict rank-0"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Atomic rank-0 save — tmp file + rename + barrier

Ex1's pattern was `if rank == 0: t.save(...); dist.barrier()`. Real training adds **atomicity** — a power loss or SIGKILL mid-`t.save` leaves a half-written file that fails to load on resume:

```python
if rank == 0:
    tmp = ckpt_path + '.tmp'
    t.save(model.state_dict(), tmp)
    os.replace(tmp, ckpt_path)   # POSIX atomic rename
dist.barrier()
```

**Why `os.replace`, not `os.rename`.** `os.replace` overwrites the destination if it exists — same semantics across POSIX and Windows. `os.rename` errors on Windows when the dest exists.

**Why the barrier still matters.** After rank 0 finishes the rename, the file is durable. But other ranks may already have charged ahead to the next step, and if subsequent code does `if rank == 1: load(ckpt)` they need to KNOW the write finished. `dist.barrier()` is the cheapest cross-rank fence — no data motion, just synchronization.

**Crash-resilience claim.** A crash AFTER `t.save(tmp)` but BEFORE `os.replace` leaves a stale `ckpt_path` and an orphaned `.tmp` file. The model can still resume from the previous good checkpoint. A crash DURING `t.save(tmp)` leaves a half-written `.tmp` file — but `ckpt_path` itself is untouched. The 'no half-written final file' guarantee is what atomic save buys you.

### Exercise 2 — atomic rank-0 save via tmp + os.replace + barrier

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Apply
> LO: Apply the `rank-0 save to tmp + os.replace + dist.barrier` pattern so the final checkpoint file never appears in a half-written state, verified by other ranks reading the file after the barrier.
> Keywords: state_dict, atomic-save, os.replace, barrier, checkpoint
> ```

**KCs targeted:** `rank0-tmp-then-rename`, `barrier-after-save-for-readers`

Implement `ex2_atomic_save(rank, world_size, dist_module, model, ckpt_path)`. Crash-resilient rank-0 save:

1. If `rank == 0`:
   a. Build `tmp_path = ckpt_path + '.tmp'`.
   b. `t.save(model.state_dict(), tmp_path)`.
   c. `os.replace(tmp_path, ckpt_path)` — POSIX atomic rename.
2. ALL ranks (including rank 0): call `dist_module.barrier()` — non-zero ranks block here until rank 0 finishes the rename.
3. Return `ckpt_path` (so the caller has the final path).

Guarantees the test verifies:
- After this function returns, `ckpt_path` exists, `ckpt_path + '.tmp'` does NOT exist.
- Loading from `ckpt_path` on any rank gives back the same state_dict that rank 0 saved.
- The barrier was actually called (so non-zero ranks didn't race ahead).

Input: `rank`, `world_size` — ints; `dist_module` — torch.distributed or mock; `model` — `nn.Module`; `ckpt_path` — str.
Output: `str` — the final checkpoint path.

In [ ]:
def ex2_atomic_save(rank: int, world_size: int, dist_module, model: 'nn.Module', ckpt_path: str) -> str:
    """Save state_dict atomically from rank 0, barrier all ranks. Return final path."""
    raise NotImplementedError()


def _test_ex2():

    import threading
    import types as _types
    import torch as _t_for_fake

    class _FakeReduceOp:
        SUM = 'SUM'
        MAX = 'MAX'
        MIN = 'MIN'
        PRODUCT = 'PROD'

    class _FakeWorld:
        """Shared state across `world_size` rank-threads."""
        def __init__(self, world_size):
            self.world_size = world_size
            self.barrier = threading.Barrier(world_size)
            self.lock = threading.Lock()
            self.scratch = {}
            self.tls = threading.local()
            self.results = [None] * world_size
            # send/recv mailbox keyed by (src, dst)
            self.mailbox = {}
            self.mailbox_cv = threading.Condition(self.lock)
        def all_reduce(self, tensor, op='SUM'):
            rank = self.tls.rank
            self.barrier.wait()
            with self.lock:
                self.scratch.setdefault('ar', [None] * self.world_size)
                self.scratch['ar'][rank] = tensor.detach().clone()
            self.barrier.wait()
            bag = self.scratch['ar']
            if op == 'SUM':
                reduced = bag[0].clone()
                for x in bag[1:]:
                    reduced = reduced + x
            elif op == 'MAX':
                reduced = bag[0].clone()
                for x in bag[1:]:
                    reduced = _t_for_fake.maximum(reduced, x)
            elif op == 'MIN':
                reduced = bag[0].clone()
                for x in bag[1:]:
                    reduced = _t_for_fake.minimum(reduced, x)
            elif op == 'PROD':
                reduced = bag[0].clone()
                for x in bag[1:]:
                    reduced = reduced * x
            else:
                raise ValueError(f'unknown fake op {op!r}')
            tensor.copy_(reduced)
            self.barrier.wait()
            if rank == 0:
                self.scratch.pop('ar', None)
            self.barrier.wait()
        def reduce(self, tensor, dst, op='SUM'):
            rank = self.tls.rank
            self.barrier.wait()
            with self.lock:
                self.scratch.setdefault('rd', [None] * self.world_size)
                self.scratch['rd'][rank] = tensor.detach().clone()
            self.barrier.wait()
            if rank == dst:
                bag = self.scratch['rd']
                if op == 'SUM':
                    reduced = bag[0].clone()
                    for x in bag[1:]:
                        reduced = reduced + x
                elif op == 'MAX':
                    reduced = bag[0].clone()
                    for x in bag[1:]:
                        reduced = _t_for_fake.maximum(reduced, x)
                elif op == 'MIN':
                    reduced = bag[0].clone()
                    for x in bag[1:]:
                        reduced = _t_for_fake.minimum(reduced, x)
                elif op == 'PROD':
                    reduced = bag[0].clone()
                    for x in bag[1:]:
                        reduced = reduced * x
                else:
                    raise ValueError(f'unknown fake op {op!r}')
                tensor.copy_(reduced)
            self.barrier.wait()
            if rank == 0:
                self.scratch.pop('rd', None)
            self.barrier.wait()
        def broadcast(self, tensor, src):
            rank = self.tls.rank
            self.barrier.wait()
            if rank == src:
                with self.lock:
                    self.scratch['bc'] = tensor.detach().clone()
            self.barrier.wait()
            if rank != src:
                tensor.copy_(self.scratch['bc'])
            self.barrier.wait()
            if rank == 0:
                self.scratch.pop('bc', None)
            self.barrier.wait()
        def barrier_op(self):
            self.barrier.wait()
        def send(self, tensor, dst):
            rank = self.tls.rank
            with self.mailbox_cv:
                self.mailbox.setdefault((rank, dst), []).append(tensor.detach().clone())
                self.mailbox_cv.notify_all()
        def recv(self, tensor, src):
            rank = self.tls.rank
            with self.mailbox_cv:
                while not self.mailbox.get((src, rank)):
                    self.mailbox_cv.wait(timeout=10)
                payload = self.mailbox[(src, rank)].pop(0)
            tensor.copy_(payload)

    def _run_fake_world(worker_fn, world_size, *extra_args, timeout=30):
        world = _FakeWorld(world_size)
        errors = [None] * world_size
        def _runner(rank):
            world.tls.rank = rank
            fake_dist = _types.SimpleNamespace()
            fake_dist.ReduceOp = _FakeReduceOp
            fake_dist.all_reduce = lambda tensor, op='SUM': world.all_reduce(tensor, op)
            fake_dist.reduce = lambda tensor, dst, op='SUM': world.reduce(tensor, dst, op)
            fake_dist.broadcast = lambda tensor, src: world.broadcast(tensor, src)
            fake_dist.barrier = world.barrier_op
            fake_dist.get_rank = lambda: rank
            fake_dist.get_world_size = lambda: world_size
            fake_dist.send = lambda tensor, dst: world.send(tensor, dst)
            fake_dist.recv = lambda tensor, src: world.recv(tensor, src)
            fake_dist.init_process_group = lambda **kw: world.scratch.setdefault('_init_calls', []).append(kw)
            fake_dist.destroy_process_group = lambda: world.scratch.setdefault('_destroy_calls', []).append(rank)
            try:
                worker_fn(rank, world_size, fake_dist, world)
            except BaseException as e:
                import traceback as _tb
                errors[rank] = (e, _tb.format_exc())
        threads = [threading.Thread(target=_runner, args=(r,), daemon=True) for r in range(world_size)]
        for th in threads:
            th.start()
        for th in threads:
            th.join(timeout=timeout)
        for r, err in enumerate(errors):
            if err is not None:
                raise RuntimeError(f'rank {r} failed: {err[0]!r}\n{err[1]}')
        return world


    import os
    import tempfile
    import torch.nn as nn

    # Use a temp dir so the test cleans up after itself.
    _tmpdir = tempfile.mkdtemp()
    _ckpt = os.path.join(_tmpdir, 'model.pt')

    def _worker(rank, world_size, dist_module, world):
        model = nn.Linear(3, 3, bias=False)
        with t.no_grad():
            model.weight.fill_(11.0)
        returned = ex2_atomic_save(rank, world_size, dist_module, model, _ckpt)
        # Read back AFTER the barrier — every rank should see the saved file.
        sd = t.load(returned, weights_only=True)
        world.results[rank] = (returned, sd['weight'].sum().item())

    w = _run_fake_world(_worker, 3)

    # Every rank returned the original path; every rank loaded weight sum = 11 * 9 = 99.
    for r in range(3):
        assert w.results[r] is not None, f'rank {r} returned None'
        path, wsum = w.results[r]
        assert path == _ckpt, f'rank {r}: returned {path}, expected {_ckpt}'
        assert abs(wsum - 99.0) < 1e-5, f'rank {r}: loaded weight sum {wsum}, expected 99'

    # Final ckpt exists; tmp does NOT (was renamed away).
    assert os.path.exists(_ckpt), 'final ckpt path must exist after atomic save'
    assert not os.path.exists(_ckpt + '.tmp'), (
        'tmp file must be renamed to final path (it should NOT linger after os.replace)'
    )

    # Verify the barrier was actually invoked — _FakeWorld.barrier uses threading.Barrier,
    # so if the student forgot the barrier the test still passes for value-correctness,
    # but we can detect it by patching dist_module.barrier and counting calls.
    # Easier: directly inspect that the loaded value is exact (which proves rank-0 finished
    # the rename before non-zero ranks loaded).

    # Re-save with a different value to confirm os.replace OVERWRITES the existing file.
    def _worker_overwrite(rank, world_size, dist_module, world):
        model = nn.Linear(3, 3, bias=False)
        with t.no_grad():
            model.weight.fill_(22.0)
        ex2_atomic_save(rank, world_size, dist_module, model, _ckpt)
        sd = t.load(_ckpt, weights_only=True)
        world.results[rank] = sd['weight'].sum().item()

    w_ov = _run_fake_world(_worker_overwrite, 2)
    for r in range(2):
        assert abs(w_ov.results[r] - 198.0) < 1e-5, (
            f'overwrite: rank {r} got {w_ov.results[r]}, expected 22*9=198'
        )

    # Detection test: count barrier calls.
    _orig_barrier = w_ov.barrier_op
    _barrier_count = [0]

    # Build a fresh world manually to instrument barrier.
    import threading as _th
    class _CountingWorld(_FakeWorld):
        def barrier_op(self):
            _barrier_count[0] += 1
            super().barrier_op()

    def _worker_count(rank, world_size, dist_module, world):
        model = nn.Linear(2, 2, bias=False)
        ex2_atomic_save(rank, world_size, dist_module, model, _ckpt)
        world.results[rank] = 'ok'

    # Run inline with the counting world.
    import types as _types
    _cw = _CountingWorld(3)
    _errs = [None] * 3
    def _instrumented_runner(rank):
        _cw.tls.rank = rank
        fake_dist = _types.SimpleNamespace()
        fake_dist.ReduceOp = _FakeReduceOp
        fake_dist.broadcast = lambda tensor, src: _cw.broadcast(tensor, src)
        fake_dist.barrier = _cw.barrier_op
        fake_dist.all_reduce = lambda tensor, op='SUM': _cw.all_reduce(tensor, op)
        fake_dist.get_rank = lambda: rank
        fake_dist.get_world_size = lambda: 3
        try:
            _worker_count(rank, 3, fake_dist, _cw)
        except BaseException as e:
            _errs[rank] = repr(e)
    _threads = [_th.Thread(target=_instrumented_runner, args=(r,), daemon=True) for r in range(3)]
    for _th_ in _threads: _th_.start()
    for _th_ in _threads: _th_.join(timeout=15)
    for r, e in enumerate(_errs):
        assert e is None, f'count run rank {r} failed: {e}'
    # Every rank called barrier ≥ once.
    assert _barrier_count[0] >= 3, (
        f'expected at least 3 barrier calls (one per rank), got {_barrier_count[0]}.  '
        f'Did you forget dist_module.barrier() on non-zero ranks?'
    )
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_atomic_save(rank: int, world_size: int, dist_module, model: 'nn.Module', ckpt_path: str) -> str:
    import os
    if rank == 0:
        tmp_path = ckpt_path + '.tmp'
        t.save(model.state_dict(), tmp_path)
        os.replace(tmp_path, ckpt_path)
    dist_module.barrier()
    return ckpt_path
```

**`os.replace` vs `os.rename`.** `os.replace` overwrites the dest if it exists, on every OS. `os.rename` errors on Windows when the dest exists. For cross-platform code, always reach for `replace`.

**Why the barrier even though only rank 0 writes.** Non-zero ranks didn't write anything, so they have nothing to wait for in ISOLATION. The barrier is for the CALLER's benefit — once `ex2_atomic_save` returns on every rank, downstream code can assume the file is durable. If a downstream `if rank == 1: load(ckpt)` ran without a prior barrier, rank 1 might attempt the load before rank 0 has finished writing.

**Atomic write + barrier are independent guarantees.** Atomicity means 'never half-written on disk'. Barrier means 'every rank agrees the write is done'. Both are needed for crash-resilient multi-rank checkpointing.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()